**Notes**:
For this example we will be using `minsearch`. This tool is a _toy tool_ tha is only used to show how can we retrieve information in a similar way that _elasticsearch_ does, but without running a Docker contained. This is not production ready code.

Alex write this code as part of the [Build a Search Engine](https://www.youtube.com/watch?v=nMrGK5QgPVE) workshop
(see the [code](https://github.com/alexeygrigorev/build-your-own-search-engine)).

In [1]:
from src.retrieve_documents import retrieve_documents
from src.searching_tool import fit_docs, search

In [2]:
## fit the text like using sklearn

documents = retrieve_documents()

index = fit_docs(documents)

Number of documents retrieved: 1342


In [3]:
## create instructions and template for the results
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [4]:
## function to create the context for the prompt

def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [5]:
## combine everything

def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [ ]:
question = "I just discovered the course. Can I join now?"
documents = retrieve_documents()
index = fit_docs(documents)
search_results = search(question, index, course="llm-zoomcamp")

prompt = build_prompt(question, search_results)

print(prompt)

Number of documents retrieved: 1342
Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capst